In [18]:
import pypsa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.grid": True,
        "grid.alpha": 0.3,
    }
)

NETWORK_PATH = (
    "/Users/fowfloi/Documents/pypsa-earth/results/cap_exp_2030/networks/elec_s_10_ec_lcopt_1h.nc"
)
n = pypsa.Network(NETWORK_PATH)
print(n)
print(f"\nSnapshots: {n.snapshots[0]} -> {n.snapshots[-1]}  ({len(n.snapshots)} hours)")
print(f"Weather year: {n.meta['load_options']['weather_year']}")
print(f"Demand year represented: 2030 (BAU growth applied to 2024 shape)")
print(f"Objective value (solved): {n.objective:,.0f}")

INFO:pypsa.io:Imported network elec_s_10_ec_lcopt_1h.nc has buses, carriers, generators, lines, line_types, loads, storage_units


PyPSA Network
Components:
 - Bus: 10
 - Carrier: 10
 - Generator: 39
 - Line: 9
 - Load: 10
 - StorageUnit: 3
Snapshots: 8784

Snapshots: 2024-01-01 00:00:00 -> 2024-12-31 23:00:00  (8784 hours)
Weather year: 2024
Demand year represented: 2030 (BAU growth applied to 2024 shape)
Objective value (solved): 3,057,044,913


In [14]:
A = pypsa.Network("results/cap_exp_2030/networks/elec_s_10_ec_lcopt_1h.nc")
B = pypsa.Network("results/cap_exp_2030_split_wacc/networks/elec_s_10_ec_lcopt_1h.nc")

for name, n in [("A", A), ("B", B)]:
    w = n.snapshot_weightings.objective
    print(f"{name}: {len(n.snapshots)} snapshots, "
          f"demand {n.loads_t.p_set.sum(axis=1).mul(w).sum()/1e6:.2f} TWh, "
          f"scale {n.meta['load_options']['scale']}")

INFO:pypsa.io:Imported network elec_s_10_ec_lcopt_1h.nc has buses, carriers, generators, lines, line_types, loads, storage_units
INFO:pypsa.io:Imported network elec_s_10_ec_lcopt_1h.nc has buses, carriers, generators, lines, line_types, loads, storage_units


A: 8784 snapshots, demand 77.78 TWh, scale 2.074
B: 8784 snapshots, demand 77.78 TWh, scale 2.074


In [21]:
built = pd.DataFrame({
    "existing_MW": n.generators.p_nom,
    "optimal_MW": n.generators.p_nom_opt,
    "carrier": n.generators.carrier,
    "bus": n.generators.bus,
})
built["new_MW"] = built.optimal_MW - built.existing_MW

print("By carrier (GW):")
print((built.groupby("carrier")[["existing_MW","optimal_MW","new_MW"]].sum()/1e3).round(2).to_string())

# where, by latitude
built["lat"] = n.buses.loc[built.bus, "y"].values
new = built[built.new_MW > 1]
print("\nNew build by bus and carrier (MW):")
print(new.pivot_table(index=["bus","lat"], columns="carrier", values="new_MW", aggfunc="sum").round(0).to_string())

By carrier (GW):
               existing_MW  optimal_MW  new_MW
carrier                                       
CCGT                  3.44        3.44    0.00
OCGT                  8.74       24.20   15.46
load shedding        13.01       13.01    0.00
onwind                0.00        0.94    0.94
solar                 0.18        5.67    5.49

New build by bus and carrier (MW):
carrier            OCGT  onwind   solar
bus   lat                              
NG0 2 6.981382      1.0     NaN     NaN
NG0 3 10.427583  7339.0     NaN     NaN
NG0 4 7.065125   8120.0     NaN     NaN
NG0 5 12.121440     NaN     NaN  3654.0
NG0 7 6.056350      1.0     NaN     NaN
NG1 0 7.628200      NaN     NaN  1193.0
NG2 0 8.171875      NaN   936.0   791.0


In [23]:
print(f"Existing line capacity: {n.lines.s_nom.sum()/1e3:.2f} GW")
print(f"Optimal line capacity : {n.lines.s_nom_opt.sum()/1e3:.2f} GW")
print(f"New: {(n.lines.s_nom_opt - n.lines.s_nom).sum()/1e3:.2f} GW")

Existing line capacity: 24.18 GW
Optimal line capacity : 24.43 GW
New: 0.24 GW


In [22]:
w = n.snapshot_weightings.objective

gen_capex = ((n.generators.p_nom_opt - n.generators.p_nom).clip(lower=0)
             * n.generators.capital_cost).sum()
gen_opex  = (n.generators_t.p.mul(w, axis=0) * n.generators.marginal_cost).sum().sum()
line_capex = ((n.lines.s_nom_opt - n.lines.s_nom).clip(lower=0) * n.lines.capital_cost).sum()

print(f"Generation capex (annualised): ${gen_capex:,.0f}")
print(f"Generation opex              : ${gen_opex:,.0f}")
print(f"Transmission capex           : ${line_capex:,.0f}")
print(f"Objective                    : ${n.objective:,.0f}")

Generation capex (annualised): $1,765,042,506
Generation opex              : $1,721,920,752
Transmission capex           : $3,798,886
Objective                    : $3,491,712,563


In [24]:
n.determine_network_topology()
print(n.buses[["x","y","sub_network"]].to_string())
print("\nlines:")
print(n.lines[["bus0","bus1"]].to_string())

               x          y sub_network
Bus                                    
NG0 0   7.414333   5.019200           0
NG0 1   7.320793   9.470453           0
NG0 2   3.745191   6.981382           0
NG0 3  11.999500  10.427583           0
NG0 4   6.891267   7.065125           0
NG0 5   7.355190  12.121440           0
NG0 6   4.721483   9.814750           0
NG0 7   5.733617   6.056350           0
NG1 0   4.155200   7.628200           1
NG2 0   9.423150   8.171875           2

lines:
       bus0   bus1
Line              
0     NG0 0  NG0 4
1     NG0 1  NG0 3
2     NG0 1  NG0 4
3     NG0 1  NG0 5
4     NG0 1  NG0 6
5     NG0 2  NG0 6
6     NG0 2  NG0 7
7     NG0 4  NG0 7
8     NG0 5  NG0 6


In [25]:
def analyze(n, label):
    connected = [b for b in n.buses.index if "sub_network" in n.buses.columns 
                 and n.buses.at[b, "sub_network"] == "0"]
    if not connected:
        n.determine_network_topology()
        connected = n.buses.index[n.buses.sub_network == n.buses.sub_network.mode()[0]].tolist()

    gens = n.generators[n.generators.bus.isin(connected)]
    built = pd.DataFrame({
        "existing_MW": gens.p_nom, "optimal_MW": gens.p_nom_opt,
        "carrier": gens.carrier, "bus": gens.bus,
    })
    built["new_MW"] = built.optimal_MW - built.existing_MW
    print(f"\n=== {label} ===")
    print((built.groupby("carrier")[["existing_MW","optimal_MW","new_MW"]].sum()/1e3).round(2).to_string())

    w = n.snapshot_weightings.objective
    gen_capex = (built.new_MW.clip(lower=0) * gens.capital_cost.values).sum()
    gen_opex = (n.generators_t.p[gens.index].mul(w, axis=0) * gens.marginal_cost).sum().sum()
    print(f"Generation capex: ${gen_capex:,.0f}   opex: ${gen_opex:,.0f}   objective: ${n.objective:,.0f}")
    return built

built_A = analyze(A, "A uniform")
built_B = analyze(B, "B split")


=== A uniform ===
               existing_MW  optimal_MW  new_MW
carrier                                       
CCGT                  3.44        3.44    0.00
OCGT                  8.74       24.18   15.44
load shedding        12.01       12.01    0.00
onwind                0.00        0.00    0.00
solar                 0.16        3.70    3.54
Generation capex: $1,106,548,191   opex: $988,430,466   objective: $3,057,044,913

=== B split ===
               existing_MW  optimal_MW  new_MW
carrier                                       
CCGT                  3.44        3.44    0.00
OCGT                  8.74       24.20   15.46
load shedding        12.01       12.01    0.00
onwind                0.00        0.00    0.00
solar                 0.16        3.67    3.51
Generation capex: $1,471,251,616   opex: $988,654,200   objective: $3,491,712,563


In [26]:
ls = n.generators[n.generators.carrier == "load shedding"]
print(f"Unserved energy: {n.generators_t.p[ls.index].mul(n.snapshot_weightings.objective, axis=0).sum().sum()/1e6:.2f} TWh")

Unserved energy: 3.15 TWh
